In [1]:
import sys
import os
sys.path.append(os.pardir)

In [2]:
import pandas as pd

In [12]:
#df = pd.read_csv("../data/raw/tempload.csv", index_col = "date", parse_dates = True)
#need to drop duplicates
#idk why the extract script is buggy
#chain it all togther
# df isn't defined yet in the df["date"] call so we need to use pipe
df = (pd.read_csv("../data/raw/tempload.csv")
        .drop_duplicates()
        .pipe(lambda df: df.set_index(pd.to_datetime(df["date"])))
        .drop(columns = ["date"])
     )

In [13]:
df.head()

,value,temperature
date,,
2023-01-01 00:00:00+00:00,28291.0,11.3615
2023-01-01 01:00:00+00:00,28727.0,9.5615
2023-01-01 02:00:00+00:00,28370.0,7.9615
2023-01-01 03:00:00+00:00,27955.0,7.3115
2023-01-01 04:00:00+00:00,27668.0,6.2615


## US Holidays

In [15]:
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar

In [17]:
cal = calendar()
holidays = cal.holidays(start = df.index.min(), end = df.index.max())

In [21]:
holidays

DatetimeIndex(['2023-01-02 00:00:00+00:00', '2023-01-16 00:00:00+00:00',
               '2023-02-20 00:00:00+00:00', '2023-05-29 00:00:00+00:00',
               '2023-06-19 00:00:00+00:00', '2023-07-04 00:00:00+00:00',
               '2023-09-04 00:00:00+00:00', '2023-10-09 00:00:00+00:00',
               '2023-11-10 00:00:00+00:00', '2023-11-23 00:00:00+00:00',
               '2023-12-25 00:00:00+00:00', '2024-01-01 00:00:00+00:00',
               '2024-01-15 00:00:00+00:00', '2024-02-19 00:00:00+00:00',
               '2024-05-27 00:00:00+00:00', '2024-06-19 00:00:00+00:00',
               '2024-07-04 00:00:00+00:00', '2024-09-02 00:00:00+00:00',
               '2024-10-14 00:00:00+00:00', '2024-11-11 00:00:00+00:00',
               '2024-11-28 00:00:00+00:00', '2024-12-25 00:00:00+00:00',
               '2025-01-01 00:00:00+00:00', '2025-01-20 00:00:00+00:00',
               '2025-02-17 00:00:00+00:00', '2025-05-26 00:00:00+00:00',
               '2025-06-19 00:00:00+00:00', '2025-0

In [19]:
df["Holiday"]= df.index.isin(holidays)

In [ ]:
#check it
df.loc[df.index.isin(holidays)]

In [35]:
df.drop(columns = ["Holiday"])

,value,temperature
date,,
2023-01-01 00:00:00+00:00,28291.0,11.361500
2023-01-01 01:00:00+00:00,28727.0,9.561500
2023-01-01 02:00:00+00:00,28370.0,7.961500
2023-01-01 03:00:00+00:00,27955.0,7.311500
2023-01-01 04:00:00+00:00,27668.0,6.261500
...,...,...
2025-12-31 19:00:00+00:00,32891.0,8.911500
2025-12-31 20:00:00+00:00,32281.0,10.361500
2025-12-31 21:00:00+00:00,31630.0,10.761499


In [33]:
#chain it in a pipe
def set_holidays(df):
    cal = calendar()
    holidays = cal.holidays(start = df.index.min(), end = df.index.max())
    df["Holiday"]= df.index.isin(holidays)
    return df

In [36]:
df = df.pipe(set_holidays)

In [37]:
df.head()

,value,temperature,Holiday
date,,,
2023-01-01 00:00:00+00:00,28291.0,11.3615,False
2023-01-01 01:00:00+00:00,28727.0,9.5615,False
2023-01-01 02:00:00+00:00,28370.0,7.9615,False
2023-01-01 03:00:00+00:00,27955.0,7.3115,False
2023-01-01 04:00:00+00:00,27668.0,6.2615,False


In [ ]:
"""
dr = pd.date_range(start='2015-07-01', end='2015-07-31')
df = pd.DataFrame()
df['Date'] = dr

cal = calendar()
holidays = cal.holidays(start=dr.min(), end=dr.max())

df['Holiday'] = df['Date'].isin(holidays)
print df
"""